# 03 · Quickstart — classify your own utterances

The minimal path: This code loads our fine-tuned model and labels the spatial words in any text you
provide. **No training, no labels required.**
Before running, click Runtime -> Change Runtime type -> change to T4 GPU.

1. Put your utterances in a CSV with an **`utterance`** column (format it like
   `data/example/test.csv`).
2. Point `INPUT_CSV` at it (default = the example).
3. Run all cells → per-word spatial predictions with calibrated confidence are written to
   `predictions.csv`. To download, click the file icon -> content -> predictions.csv

Model: [`SamAgnoli/deberta-v3-base-spatial-language-detection`](https://huggingface.co/SamAgnoli/deberta-v3-base-spatial-language-detection).


In [ ]:
!pip install -q transformers torch huggingface_hub pandas numpy

In [ ]:
# === Setup: locate the model's data files (matcher, calibration, example) ===
# Works in Colab (auto-clones the repo) and locally (uses the repo you're already in).
import os, re, json, string

def _find_data_root():
    for d in ("../data", "data", "spatial-language-classifier/data",
              "/content/spatial-language-classifier/data"):
        if os.path.isdir(d):
            return os.path.abspath(d)
    return None

DATA_ROOT = _find_data_root()
if DATA_ROOT is None:                       # fresh Colab: fetch the repo
    os.system("git clone -q https://github.com/SamAgnoli/spatial-language-classifier.git")
    DATA_ROOT = _find_data_root()
assert DATA_ROOT, "Couldn't find the repo's data/ folder \u2014 is the repo public on GitHub?"

# --- Infrastructure (tied to the published model \u2014 leave these as-is) ---
MODEL_SOURCE = os.environ.get("SPATIAL_MODEL",   "SamAgnoli/deberta-v3-base-spatial-language-detection")
MATCHER_PATH = os.environ.get("SPATIAL_MATCHER", os.path.join(DATA_ROOT, "spatial_matcher.json"))
CALIB_PATH   = os.environ.get("SPATIAL_CALIB",   os.path.join(DATA_ROOT, "calibration.json"))

# --- YOUR INPUT: a CSV with an `utterance` column ----------------------------------
# Default = the bundled example. To classify YOUR OWN data in Colab: open the file panel
# (folder icon, left), upload your CSV, then change the line below to e.g.:
#     INPUT_CSV = "/content/my_utterances.csv"
INPUT_CSV  = os.environ.get("SPATIAL_INPUT",  os.path.join(DATA_ROOT, "example", "test.csv"))
OUTPUT_CSV = os.environ.get("SPATIAL_OUTPUT", "predictions.csv")

print("DATA_ROOT =", DATA_ROOT)
print("INPUT_CSV =", INPUT_CSV)


In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_SOURCE).to(device).eval()

# Calibration temperature fitted on the original validation set (section 7.5 of the
# training notebook). Falls back to 1.0 (raw probabilities) if not present.
try:
    with open(CALIB_PATH, 'r', encoding='utf-8') as _f:
        T = float(json.load(_f)['temperature_T'])
except Exception:
    T = 1.0
    print('No calibration.json found; using T = 1.0 (uncalibrated).')
print('Model loaded on', device, '| temperature T =', T)


In [ ]:
# --- Dictionary matcher ---
with open(MATCHER_PATH, 'r', encoding='utf-8') as _f:
    _m = json.load(_f)
EXACT = set(_m['EXACT'])
PREFIXES = list(_m['PREFIX'])

_STRIP = string.punctuation + ' \t\n'

def is_candidate(word: str) -> bool:
    w = word.lower().strip(_STRIP)
    if not w:
        return False
    if w in EXACT:
        return True
    for stem in PREFIXES:
        if w.startswith(stem):
            return True
    return False

print('Matcher: %d EXACT, %d PREFIX entries' % (len(EXACT), len(PREFIXES)))


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device).eval()

WORD_RE = re.compile(r"[A-Za-z']+")  # words only; skips punctuation and digits

def classify_utterance(utterance: str, batch_size: int = 32):
    """Return per-word spatial labels for dictionary-candidate words only.

    Non-candidate words (e.g. 'what', 'did', 'we') are omitted from the output.
    Each result carries both the raw model probability and the *calibrated*
    confidence (from temperature T fitted in section 7.5) so unsure predictions
    are visible at inference time. The hard 0/1 decision is unchanged — it's
    the model's raw argmax — so live behavior matches the test metrics.
    """
    tokens = WORD_RE.findall(utterance)
    cands = [(i, w) for i, w in enumerate(tokens) if is_candidate(w)]
    if not cands:
        return []
    positions = [i for i, _ in cands]
    words = [w for _, w in cands]
    results = []
    with torch.no_grad():
        for start in range(0, len(words), batch_size):
            end = start + batch_size
            chunk_w = words[start:end]
            chunk_p = positions[start:end]
            enc = tokenizer(
                [utterance] * len(chunk_w),
                chunk_w,
                truncation=True,
                max_length=128,
                padding=True,
                return_tensors='pt',
            ).to(device)
            logits = model(**enc).logits                                # raw logits
            raw_probs = torch.softmax(logits, dim=-1).cpu().numpy()
            z = (logits[:, 1] - logits[:, 0]).cpu().numpy()             # binary decision logit
            cal_p = 1.0 / (1.0 + np.exp(-z / T))                        # calibrated P(spatial)
            for pos, w, raw_row, p_cal in zip(chunk_p, chunk_w, raw_probs, cal_p):
                results.append({
                    'word': w,
                    'position': pos,
                    'spatial_or_not': int(raw_row.argmax()),
                    'prob_spatial_raw':    float(raw_row[1]),
                    'confidence_spatial':  float(p_cal),
                    'confidence_decision': float(max(p_cal, 1.0 - p_cal)),
                })
    return results

def print_predictions(utterance: str):
    print(f'Utterance: {utterance}')
    preds = classify_utterance(utterance)
    if not preds:
        print('  (no dictionary-candidate words)')
        return
    for r in preds:
        flag = '[SPATIAL]' if r['spatial_or_not'] == 1 else '[       ]'
        # confidence_decision near 0.5 = unsure; near 1.0 = confident either way
        unsure = ' ← unsure' if r['confidence_decision'] < 0.65 else ''
        print(f"  {flag} pos={r['position']:2d}  {r['word']:15s}  "
              f"p_spatial={r['confidence_spatial']:.3f}  "
              f"confidence={r['confidence_decision']:.3f}{unsure}")


In [ ]:
# --- Apply to your CSV of utterances ---
import pandas as pd

inp = pd.read_csv(INPUT_CSV)
assert 'utterance' in inp.columns, "Input CSV must have an 'utterance' column."
utterances = inp['utterance'].dropna().astype(str).drop_duplicates().tolist()
print(f'Classifying {len(utterances)} unique utterances from {INPUT_CSV} ...')

rows = []
for utt in utterances:
    for r in classify_utterance(utt):
        rows.append({'utterance': utt, **r})

pred_df = pd.DataFrame(rows)
pred_df.to_csv(OUTPUT_CSV, index=False)
print(f'Wrote {len(pred_df)} per-word predictions -> {OUTPUT_CSV}')
pred_df.head(20)
